In [1]:
!pip show pillow | grep Version

Version: 11.3.0


In [2]:
!pip install -q roboflow
!pip install -q rfdetr
!pip install -q inference

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 285.0/285.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 486.3/486.3 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.7/102.7 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.2/280.2 kB 11.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.7/105.7 kB 1.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 2.6 MB/s eta 

In [4]:
# Célula 2 — rodar SOMENTE depois de reiniciar o runtime
import PIL
print(PIL.__version__)

from rfdetr import RFDETRSmall
from roboflow import Roboflow
import supervision as sv

11.3.0


ImportError: cannot import name '_Ink' from 'PIL._typing' (/usr/local/lib/python3.12/dist-packages/PIL/_typing.py)

In [ ]:
import glob
import os

# Dynamically find all .jpg files in the /content/ directory
imagens = glob.glob('/content/*.jpg')

# Sort the list for consistent processing order
imagens.sort()

print(f"Found {len(imagens)} image files in /content/:")
for i, img_path in enumerate(imagens):
    print(f"  {i+1}. {img_path}")

In [ ]:
from PIL import Image
from inference import get_model
import numpy as np
import supervision as sv
import matplotlib.pyplot as plt
import os
import glob

model = get_model("guilhermes-workspace-mmoyg/modelo-yolo-v12-v11-cajazeiras-6-yolo11n-t1", api_key="00ivd23tUucrDPOWwuTd")

# Lista '1 a 1' atualizada com os arquivos encontrados em /content/
imagens = glob.glob('/content/*.jpg')

box_annotator   = sv.BoxAnnotator(thickness=10)
label_annotator = sv.LabelAnnotator(text_scale=10.5, text_thickness=5, text_padding=5)

for caminho in imagens:
    if not os.path.exists(caminho):
        print(f"Arquivo não encontrado: {caminho}")
        continue

    img        = Image.open(caminho)
    image_np   = np.array(img)

    predictions = model.infer(img, confidence=0.15)[0]
    detections  = sv.Detections.from_inference(predictions)

    labels = [f"Fachada {conf:.0%}" for conf in detections.confidence]

    annotated = box_annotator.annotate(scene=image_np.copy(), detections=detections)
    annotated = label_annotator.annotate(scene=annotated, detections=detections, labels=labels)

    plt.figure(figsize=(20, 20), dpi=150)
    plt.title(os.path.basename(caminho))
    plt.imshow(annotated)
    plt.axis("off")
    plt.show()

    print(f"{os.path.basename(caminho)}: {len(detections)} detecção(ões)")

In [ ]:
print("Caminhos das imagens atualizados:")
for i, img_path in enumerate(imagens):
    print(f"  {i+1}. {img_path}")
print(f"Total de imagens: {len(imagens)}")

In [ ]:
for caminho in imagens:
    img = Image.open(caminho)
    predictions = model.infer(img, confidence=0.65)[0]

    print(f"\n{'='*60}")
    print(f"Imagem: {caminho.split('/')[-1]}")
    print(f"Total de detecções: {len(predictions.predictions)}")
    for i, pred in enumerate(predictions.predictions):
        print(f"  [{i+1}] classe={pred.class_name} | conf={pred.confidence:.2%} "
              f"| centro=({pred.x:.0f}, {pred.y:.0f}) "
              f"| tamanho=({pred.width:.0f}x{pred.height:.0f}px)")
    print(f"{'='*60}")

In [ ]:
import psycopg2